# V025: conservative validation-only pruning

Only native point-weighted inspiratory validation MAE on the same grouped split enters the stopping rule. Best-so-far validation MAE protects temporary spikes. New variants/schedules/data sizes/runtimes without comparable completed history finish a representative. Comparisons begin halfway through the budget, use matched optimizer steps and LR prefixes, and require a sustained deficit against even the weakest comparable reference plus stalled improvement.

Calibration uses actual completed curves: twice the largest late deficit recovered by a later-winning run sets the relative margin; the largest observed material-improvement gap sets patience; the largest observed post-reduction recovery delay sets LR grace. Material improvement is measured against empirical late-quarter validation variation. No absolute MAE cutoff, training error or leaderboard metric is used. The runtime training wrapper must verify a resumable checkpoint before acting.

In [ ]:
"""Conservative, validation-only early pruning. No model/data fitting here."""
import json,hashlib,math,copy
import numpy as np

def pruning_digest_v025(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def major_LR_reductions_v025(rates):
    rates=np.asarray(rates,dtype=np.float64)
    if rates.ndim!=1 or len(rates)==0 or not np.isfinite(rates).all() or (rates<0).any():
        raise ValueError('Finite nonnegative complete LR schedule required')
    peak=int(np.argmax(rates));anchor=float(rates[peak]);events=[]
    for j in range(peak+1,len(rates)):
        if anchor>0 and rates[j]<=.5*anchor:
            events.append(j+1);anchor=float(rates[j])
    return events

def calibrate_pruning_v025(cohort,references,known_variants):
    """Use completed same-stage validation curves; never training errors/leaderboards."""
    if len(references)<3:raise ValueError('Finish at least3 comparable representatives before calibration')
    budget=int(cohort['budget_epochs']);half=math.ceil(budget/2);curves=[];late_start=math.ceil(.75*budget)-1
    for ref in references:
        v=np.asarray(ref['validation_inspiratory_MAE'],np.float64)
        b=np.asarray(ref['best_validation_inspiratory_MAE'],np.float64)
        steps=np.asarray(ref['optimizer_steps'],np.int64);rates=np.asarray(ref['LR'],np.float64)
        if not (len(v)==len(b)==len(steps)==len(rates)==budget and np.isfinite(v).all() and np.isfinite(b).all() and (b>0).all()):
            raise ValueError('Complete finite positive validation histories required')
        if not (np.diff(steps)>0).all() or not (np.diff(b)<=0).all():raise ValueError('Progress/best-metric history changed')
        if not np.array_equal(rates,np.asarray(cohort['planned_LR_schedule'])):raise ValueError('Unmatched LR schedule')
        curves.append((ref,v,b,steps,rates))
    noise=max(float(np.quantile(np.abs(np.diff(v[late_start:]))/v[late_start:-1],.9)) for _,v,_,_,_ in curves)
    if not (0<noise<1):raise ValueError('Insufficient empirical metric variation for this calibration')
    recovered=[]
    for a,_,aa,_,_ in curves:
        for b,_,bb,_,_ in curves:
            if a['id']!=b['id'] and aa[-1]<=bb[-1]:
                relative=np.maximum((aa[half-1:]-bb[half-1:])/bb[half-1:],0)
                at=int(np.argmax(relative))
                recovered.append(dict(run=a['id'],reference=b['id'],relative_deficit=float(relative[at]),epoch=half+at))
    maximum_recovered=max((a['relative_deficit'] for a in recovered),default=0.)
    if maximum_recovered<=0:raise ValueError('No recovered-deficit evidence; finish more representatives')
    details=[];all_gaps=[];all_delays=[]
    for ref,_,best,_,rates in curves:
        anchor=float(best[half-1]);last=half;gaps=[]
        for j in range(half,budget):
            if best[j]<anchor*(1-noise):
                gaps.append(j+1-last);last=j+1;anchor=float(best[j])
        delays=[];censored=[]
        for epoch in major_LR_reductions_v025(rates):
            recovered_at=np.flatnonzero(best[epoch-1:]<best[epoch-2]*(1-noise))
            if len(recovered_at):delays.append(int(recovered_at[0]+1))
            else:censored.append(epoch)
        all_gaps.extend(gaps);all_delays.extend(delays)
        details.append(dict(id=ref['id'],material_improvement_gaps=gaps,major_LR_reduction_epochs=major_LR_reductions_v025(rates),
            material_recovery_delays=delays,censored_reductions=censored))
    if not all_gaps:raise ValueError('No material-improvement gaps to calibrate patience')
    patience=max(2,max(all_gaps));cooldown=max(all_delays,default=0)
    return dict(schema='v025-validation-only-pruning-policy-v1',enabled=True,
        metric='native_point_weighted_validation_inspiratory_MAE',cohort=copy.deepcopy(cohort),
        cohort_sha256=pruning_digest_v025(cohort),known_variants=copy.deepcopy(known_variants),
        references=copy.deepcopy(references),minimum_comparable_references=2,
        start_comparing_epoch=half,patience_intervals=patience,LR_cooldown_evaluations=cooldown,
        relative_deficit_margin=2*maximum_recovered,stall_relative_tolerance=noise,
        reference_envelope='highest_same_stage_best_validation_MAE_across_comparable_completed_references',
        calibration=dict(reference_count=len(references),recovered_deficits=recovered,
            largest_recovered_relative_deficit=maximum_recovered,details=details,
            margin_rule='twice largest observed post-halfway deficit that a later-winning reference recovered',
            patience_rule='largest observed interval between material validation improvements',
            cooldown_rule='largest observed delay from a half-rate crossing to material validation improvement',
            stall_rule='largest per-run90th percentile of late-quarter relative validation changes'),
        limitations='Small completed-history sample. Only known model variants on exactly matched data/optimizer/LR/runtime are eligible; unfamiliar variants/schedules/splits finish a representative.')

def evaluate_pruning_v025(history,descriptor,policy):
    """Stateless replayable decision from validation fields and progress only."""
    def keep(reason,**extra):return dict(action='continue',reason=reason,**extra)
    if not policy.get('enabled'):return keep('policy_not_enabled')
    if descriptor.get('cohort')!=policy['cohort']:return keep('no_comparable_data_model_optimizer_schedule_or_runtime')
    variant=descriptor.get('variant')
    if variant not in policy['known_variants']:return keep('unfamiliar_variant_finish_representative')
    expected=policy['known_variants'][variant]
    identity=descriptor.get('factory_identity',{})
    if identity.get('code_sha256') not in expected['allowed_factory_code_sha256']:
        return keep('unfamiliar_factory_finish_representative')
    sources=descriptor.get('model_source_hashes',{})
    if any(sources.get(k)!=v for k,v in expected['model_source_hashes'].items()):
        return keep('unfamiliar_model_implementation_finish_representative')
    if not history:return keep('no_validation_history')
    try:
        e=np.array([int(r['epoch']) for r in history]);steps=np.array([int(r['cumulative_optimizer_steps']) for r in history])
        best=np.array([float(r['best_val_mae']) for r in history]);rates=np.array([float(r['lr_used']) for r in history])
    except (KeyError,TypeError,ValueError):return keep('untrusted_validation_progress_fields')
    if not (np.array_equal(e,np.arange(1,len(e)+1)) and np.isfinite(best).all() and (best>0).all() and (np.diff(best)<=0).all()):
        return keep('untrusted_or_nonfinite_validation_history')
    epoch=int(e[-1]);budget=int(policy['cohort']['budget_epochs'])
    if epoch>=budget:return keep('training_budget_complete',epoch=epoch)
    if epoch<policy['start_comparing_epoch']:return keep('warmup_before_half_budget',epoch=epoch)
    refs=[]
    for ref in policy['references']:
        if ref['id']==descriptor.get('run_id'):continue
        if len(ref['optimizer_steps'])>=epoch and np.array_equal(steps,np.asarray(ref['optimizer_steps'][:epoch])) and np.array_equal(rates,np.asarray(ref['LR'][:epoch])):
            refs.append(ref)
    if len(refs)<policy['minimum_comparable_references']:return keep('insufficient_same_stage_step_and_LR_references',epoch=epoch)
    p=int(policy['patience_intervals']);first=epoch-p
    if first<policy['start_comparing_epoch']:return keep('collecting_post_halfway_patience',epoch=epoch)
    reductions=major_LR_reductions_v025(policy['cohort']['planned_LR_schedule'])
    # Include an immediately upcoming scheduled half-rate change before allowing a stop.
    recent=max((r for r in reductions if r<=epoch+1),default=None)
    if recent is not None and epoch-recent<policy['LR_cooldown_evaluations']:
        return keep('LR_reduction_grace',epoch=epoch,latest_major_reduction_epoch=recent)
    sl=slice(first-1,epoch)
    envelopes=np.max(np.stack([np.asarray(r['best_validation_inspiratory_MAE'])[sl] for r in refs]),axis=0)
    gap=best[sl]/envelopes-1
    improvement=float((best[first-1]-best[-1])/best[first-1])
    substantially_behind=bool((gap>policy['relative_deficit_margin']).all())
    stalled=bool(improvement<=policy['stall_relative_tolerance'])
    details=dict(epoch=epoch,optimizer_steps=int(steps[-1]),best_validation_inspiratory_MAE=float(best[-1]),
        window_start_epoch=first,patience_intervals=p,relative_best_validation_improvement=improvement,
        minimum_relative_deficit_in_window=float(gap.min()),relative_deficit_margin=policy['relative_deficit_margin'],
        stall_relative_tolerance=policy['stall_relative_tolerance'],substantially_behind=substantially_behind,
        validation_improvement_stalled=stalled,reference_ids=[r['id'] for r in refs],
        reference_stage_best_validation_MAE={r['id']:float(r['best_validation_inspiratory_MAE'][epoch-1]) for r in refs})
    if substantially_behind and stalled:return dict(action='prune',reason='sustained_same_stage_validation_deficit_and_stalled_improvement',**details)
    return keep('within_reference_envelope' if not substantially_behind else 'validation_still_improving',**details)
